# Setup

In [1]:
from pprint import pprint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# sklearn config
from sklearn import set_config
set_config(transform_output='pandas', display='text')

# Notation
  - $X \in \mathbb{R}^{m\times n}$: input data; $Y \in \mathbb{R}^{m\times k}$: reduced data.
  - $m$ = n_samples
  - $n$ = n_features
  - $k$ = n_components

# 1. PCA

## 1) Summary

### Algorithm

- PCA, Principal Component Analysis
  - Linear projection onto orthogonal directions with the largest variance.

1. Center each feature: $X_c=X-\mu$, where $\mu$ = element-wise sample mean.
2. Calculate the covariance matrix: $S=\frac{X_c^\top X_c}{m-1}$.
3. Eigen decomposition: $Sv_j=\lambda_jv_j$.
    - $v_j$: unit eigenvector (direction).
    - $\lambda_j$: variance along that direction.
    - Sort $\lambda_1\ge\cdots\ge\lambda_n$ and collect the first $k$ orthogonal eigenvectors as columns of $V_k$.
4. Project: $Y=X_cV_k$.

### Parameters / Evaluation

- `n_components`
  - Number of directions to keep.
- Explained variance ratio
  - $r_j=\frac{\lambda_j}{\sum_{\ell=1}^{n}\lambda_\ell}$, available as `explained_variance_ratio_`.
  - Choose the smallest $k$ reaching the target cumulative ratio, e.g. $\sum_{j=1}^{k}r_j\ge0.95$.
- SVD, Singular Value Decomposition
  - Equivalent calculation on centered data: $X_c=U\Sigma V^\top$.
  - The first $k$ columns of $V$ give the same projection directions.

### Practical tips

- Usage: linear compression of numerical features.
- Scaling
  - PCA centers features, but does not scale them.
  - Scale first when feature units / ranges differ.
- New samples
  - $Y_{\mathrm{new}}=(X_{\mathrm{new}}-\mu)V_k$.
  - `transform` reuses the fitted mean and directions.

## 2) API

```python
from sklearn.decomposition import PCA

class PCA(
    n_components: Int | Float | Literal['mle'] | None = None,
    *,
    copy: bool = True,
    whiten: bool = False,
    svd_solver: Literal['auto', 'full', 'covariance_eigh', 'arpack', 'randomized'] = "auto",
    tol: Float = 0.0,
    iterated_power: Int | Literal['auto'] = "auto",
    n_oversamples: Int = 10,
    power_iteration_normalizer: Literal['auto', 'QR', 'LU', 'none'] = "auto",
    random_state: Int | RandomState | None = None
)
```

# 2. Truncated SVD

## 1) Summary

### Algorithm

- Truncated SVD, Truncated Singular Value Decomposition

1. Use $X$ without centering.
2. Calculate the largest $k$ singular values and their singular vectors.
    - $X\approx U_k\Sigma_kV_k^\top$.
    - $\Sigma_k$: diagonal matrix of retained singular values.
    - $V_k$: retained directions.
3. Project: $Y=XV_k$.

### Parameters / Evaluation

- `n_components`
  - Number of directions to keep.
  - Compare cumulative `explained_variance_ratio_` and downstream validation performance.

### Practical tips

- Sparse data
  - Most entries are zero.
  - No centering $\rightarrow$ the input can remain sparse.
- PCA vs Truncated SVD
  - PCA applies SVD to centered data; Truncated SVD uses the original data.
- LSA, Latent Semantic Analysis
  - Truncated SVD on a term-count / TF-IDF (Term Frequency-Inverse Document Frequency) matrix.
- New samples
  - $Y_{\mathrm{new}}=X_{\mathrm{new}}V_k$.
  - `transform` reuses the fitted directions.

## 2) API

```python
from sklearn.decomposition import TruncatedSVD

class TruncatedSVD(
    n_components: Int = 2,
    *,
    algorithm: Literal['arpack', 'randomized'] = "randomized",
    n_iter: Int = 5,
    n_oversamples: Int = 10,
    power_iteration_normalizer: Literal['auto', 'QR', 'LU', 'none'] = "auto",
    random_state: Int | RandomState | None = None,
    tol: Float = 0.0
)
```

# 3. t-SNE

## 1) Summary

### Algorithm

- t-SNE, t-distributed Stochastic Neighbor Embedding
  - Preserve local neighbor relationships.
  - The formulas below use a 2D output.

1. Calculate high-dimensional neighbor probabilities, for $i\ne j$.
    - $p_{j\mid i}=\frac{\exp(-\|x_i-x_j\|^2/(2\sigma_i^2))}{\sum_{\ell\ne i}\exp(-\|x_i-x_\ell\|^2/(2\sigma_i^2))}$.
    - Choose each $\sigma_i$ by binary search so that $2^{-\sum_{j\ne i}p_{j\mid i}\log_2p_{j\mid i}}=\mathrm{perplexity}$.
2. Symmetrize: $p_{ij}=\frac{p_{j\mid i}+p_{i\mid j}}{2m}$.
3. Init the low-dimensional coordinates $Y$ using PCA or small random values.
4. Calculate low-dimensional neighbor probabilities.
    - $q_{ij}=\frac{(1+\|y_i-y_j\|^2)^{-1}}{\sum_{a\ne b}(1+\|y_a-y_b\|^2)^{-1}}$.
5. Update $Y$ by gradient descent to reduce the Kullback-Leibler divergence.
    - $L=\sum_{i\ne j}p_{ij}\log\frac{p_{ij}}{q_{ij}}$.
    - Keep $p_{ij}$ fixed; recalculate $q_{ij}$ as $Y$ changes.
6. Repeat 4-5 until `max_iter`, insufficient progress, or a sufficiently small gradient.

### Parameters / Evaluation

- `n_components`
  - Usually 2 or 3 for visualization.
- `perplexity`
  - Effective neighborhood size, not a fixed nearest-neighbor count.
  - Smaller $\rightarrow$ more local detail.
  - Larger $\rightarrow$ broader neighborhoods.
  - Must be smaller than $m$.
- `random_state`
  - Fix the seed for repeatability.
  - Compare different seeds / perplexities for stability.

### Practical tips

- Usage: mainly 2D / 3D visualization, not a general-purpose compression step.
- Interpretation
  - Distances between groups need not match distances in the original space.
- New samples
  - No `transform` in sklearn.
  - `fit_transform` returns coordinates for the samples used in that fit.

## 2) API

```python
from sklearn.manifold import TSNE

class TSNE(
    n_components: Int = 2,
    *,
    perplexity: Float = 30.0,
    early_exaggeration: Float = 12.0,
    learning_rate: Float | Literal['auto'] = "auto",
    max_iter: Int = 1000,
    n_iter_without_progress: Int = 300,
    min_grad_norm: Float = 1e-7,
    metric: str | Callable = "euclidean",
    metric_params: dict | None = None,
    init: Literal['pca', 'random'] | ArrayLike = "pca",
    verbose: Int = 0,
    random_state: Int | RandomState | None = None,
    method: Literal['barnes_hut', 'exact'] = "barnes_hut",
    angle: Float = 0.5,
    n_jobs: Int | None = None
)
```

# 4. UMAP

## 1) Summary

### Algorithm

- UMAP, Uniform Manifold Approximation and Projection
  - Manifold learning: find a low-dimensional structure within high-dimensional data.
  - The steps below use the default local connectivity and graph union.

1. Find $r=\mathrm{n\_neighbors}$ nearest samples for each $x_i$.
    - The search includes itself; exclude self-links from the neighbor set $N_i$.
2. Convert neighbor distances into directed connection strengths.
    - $a_{ij}=\exp(-\max(0,d(x_i,x_j)-\rho_i)/\sigma_i)$, for $j\in N_i$; otherwise $a_{ij}=0$.
    - $\rho_i$: nearest positive neighbor distance.
    - Choose the local scale $\sigma_i$ so that $\sum_{j\in N_i}a_{ij}\approx\log_2r$.
3. Combine both directions: $w_{ij}=a_{ij}+a_{ji}-a_{ij}a_{ji}$.
4. Init the low-dimensional coordinates $Y$.
5. Define low-dimensional connection strengths: $v_{ij}=\frac{1}{1+a\|y_i-y_j\|^{2b}}$.
    - $a,b$: curve constants determined by `min_dist` and `spread`.
6. Repeatedly update $Y$ for `n_epochs`.
    - Sample neighbor pairs according to $w_{ij}$ and reduce $-\log v_{ij}$ to pull them together.
    - Sample random pairs and reduce $-\log(1-v_{ij})$ to push them apart.

### Parameters / Evaluation

- `n_neighbors`
  - Smaller $\rightarrow$ local detail.
  - Larger $\rightarrow$ broader structure.
- `min_dist`
  - Smaller $\rightarrow$ tighter groups.
  - Larger $\rightarrow$ more spread-out points.
- `n_components`
  - 2 / 3 for visualization; higher values for downstream models.
- `random_state`
  - Fix the seed for repeatability.
  - Compare multiple settings for stability.

### Practical tips

- Usage: nonlinear visualization / dimensionality reduction.
- New samples
  - `transform` places new samples in the fitted low-dimensional space.
- Interpretation
  - Group spacing and density can be distorted.
  - A separated group in the plot is not proof of a real cluster.
- Package
  - `umap-learn`, not a built-in sklearn estimator.

## 2) API

```python
from umap import UMAP

# Main parameters only; other options omitted.
class UMAP(
    n_neighbors: Int = 15,
    n_components: Int = 2,
    metric: str | Callable = "euclidean",
    n_epochs: Int | None = None,
    learning_rate: Float = 1.0,
    init: Literal['spectral', 'random', 'pca', 'tswspectral'] | ArrayLike = "spectral",
    min_dist: Float = 0.1,
    spread: Float = 1.0,
    n_jobs: Int = -1,
    random_state: Int | RandomState | None = None,
    verbose: bool = False
)
```

# Evaluation

- Predictive performance
  - Preserving 95% of variance does not mean preserving 95% of predictive information.
  - Compare the final model's validation performance with / without dimensionality reduction.
- Data leakage
  - Fit scaling and dimensionality reduction on train only.
  - Reuse the fitted transformations for validation / test.
  - Use a `Pipeline` to repeat this correctly within cross-validation.
- Visualization
  - Check whether the structure remains similar across settings / repetitions.
  - Do not choose a method only because its plot looks well-separated.